## Topic: Population vs Sample

**Population** = everything you care about. Every possible data point.
**Sample** = the part of it you actually managed to collect.

Examples:
- Population: all adults in India. Sample: the 2,000 people you surveyed.
- Population: every trade that will ever happen. Sample: last year's price data.
- Population: all possible photos of cats. Sample: the 50,000 in your training set.

### Why this matters
You almost never have the population. Collecting everything is too slow,
too expensive, or literally impossible (you can't survey future customers).

So you work with a sample and try to say something about the whole. That's the
entire job of statistics — reasoning about the population from a sample.

### The catch
Your sample is only a *guess* at the population, so there's always some
uncertainty. Two different samples give two slightly different answers.

Statistics is largely about answering: how much can I trust what my sample
is telling me?

### A word on notation
You'll see different symbols for the same thing depending on whether we mean
the population or the sample:

- population mean = μ (mu), sample mean = x̄ (x-bar)
- population std dev = σ (sigma), sample std dev = s

Same concept, but one is the truth and the other is our estimate of it.

### The main trap: a bad sample
A sample only works if it actually represents the population.

Classic failure: surveying people about smartphone habits... by phoning them
on a landline. You've quietly excluded everyone who only owns a mobile.
That's **sampling bias**, and no amount of maths fixes it. Bigger sample,
same bias, more confidently wrong.

In ML the same thing happens: if your training data doesn't represent the real
world, the model fails when deployed — no matter how good the training metrics look.

## Topic 2: Descriptive Statistics

You've got a pile of numbers. Descriptive statistics is just summarising that
pile so a human can understand it.

Nobody can read 10,000 salaries. But you can say: "typical salary is £45k, most
people are between £35k and £60k, a few earn much more." That's the job.


### Three ways to say "the middle"

- **Mean** — the ordinary average. Add everything, divide by how many.
- **Median** — sort the numbers, take the middle one.
- **Mode** — the value that shows up most often.

For nice balanced data these end up similar. The interesting bit is when they don't.

In [3]:
numbers = [2, 4, 4, 8, 12]

# MEAN - add everything up, divide by how many
total = 0
for n in numbers:
    total = total + n
mean = total / len(numbers)
print("mean =", mean)

# MEDIAN - sort, then take the middle value
sorted_numbers = sorted(numbers)
middle_position = len(sorted_numbers) // 2

if len(sorted_numbers) % 2 == 1:          # odd count -> one middle value
    median = sorted_numbers[middle_position]
else:                                      # even count -> average the two middles
    median = (sorted_numbers[middle_position - 1] + sorted_numbers[middle_position]) / 2
print("median =", median)

# MODE - count how many times each value appears, pick the most common
counts = {}
for n in numbers:
    if n in counts:
        counts[n] = counts[n] + 1
    else:
        counts[n] = 1

mode = None
highest_count = 0
for value in counts:
    if counts[value] > highest_count:
        highest_count = counts[value]
        mode = value
print("mode =", mode, "- it appeared", highest_count, "times")

mean = 6.0
median = 4
mode = 4 - it appears 2 times


### Why the median is often better than the mean

The mean gets dragged around by extreme values. The median doesn't.

Take five people in a room and look at their salaries. Then let a billionaire
walk in and watch what happens to each number.

In [4]:
# small helper functions so we can reuse them
def get_mean(nums):
    total = 0
    for n in nums:
        total = total + n
    return total / len(nums)

def get_median(nums):
    s = sorted(nums)
    mid = len(s) // 2
    if len(s) % 2 == 1:
        return s[mid]
    else:
        return (s[mid - 1] + s[mid]) / 2


salaries = [30, 35, 40, 45, 50]        # in thousands
print("salaries:", salaries)
print("  mean   =", get_mean(salaries))
print("  median =", get_median(salaries))

# now a billionaire joins the room
salaries_with_boss = [30, 35, 40, 45, 50, 1000000]
print("\nafter the billionaire arrives:", salaries_with_boss)
print("  mean   =", get_mean(salaries_with_boss))
print("  median =", get_median(salaries_with_boss))

salaries: [30, 35, 40, 45, 50]
  mean   = 40.0
  median = 40

after the billionaire arrives: [30, 35, 40, 45, 50, 1000000]
  mean   = 166700.0
  median = 42.5


The mean jumped to about 166,700 — a number that describes **nobody** in the room.
The median only moved from 40 to 42.5.

That's because the median only cares about *position* in the sorted list, not
how extreme a value is. One giant number can only ever be "the last one".

This is why news reports say "median house price" and "median salary", never mean.
A few mansions would make the mean meaningless.

**Rule of thumb:** if the mean and median are far apart, your data is skewed
or has outliers. That gap is itself a useful signal.

### Percentiles and quartiles

A percentile tells you where a value sits in the ranking.
The 90th percentile is the value that 90% of the data falls below.
(If your exam score is at the 90th percentile, you beat 90% of people.)

Quartiles just cut the data into four equal chunks:

- Q1 = 25th percentile — a quarter of the data is below this
- Q2 = 50th percentile — this is the median
- Q3 = 75th percentile — three quarters below this

**IQR (interquartile range) = Q3 − Q1**

That's the range covering the middle 50% of the data. It's useful because it
describes the typical bulk of your data while ignoring the extremes completely.

### Spotting outliers

An outlier is a value sitting far away from everything else.

The usual rule (it's what boxplots use):

    anything below  Q1 − 1.5 × IQR
    or above        Q3 + 1.5 × IQR

Think of these two numbers as a fence around your data. Anything outside the
fence gets flagged.

Note: if your data is well behaved, the fences will sit *outside* the actual
data range. That's not a mistake — it just means nothing is extreme enough to
flag. A fence cutting through your data would be flagging normal values.

In [6]:
import numpy as np

def find_outliers(data):
    q1 = np.percentile(data, 25)
    q3 = np.percentile(data, 75)
    iqr = q3 - q1

    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr

    print("data:", data)
    print("  Q1 =", q1, "| Q3 =", q3, "| IQR =", iqr)
    print("  fences:", lower_fence, "to", upper_fence)
    print("  actual data range:", min(data), "to", max(data))

    outliers = []
    for value in data:
        if value < lower_fence or value > upper_fence:
            outliers.append(value)

    if len(outliers) == 0:
        print("  outliers: none")
    else:
        print("  outliers:", outliers)
    print()


find_outliers([10, 20, 30, 40, 50])     # well behaved
find_outliers([10, 20, 30, 40, 500])    # one extreme value

data: [10, 20, 30, 40, 50]
  Q1 = 20.0 | Q3 = 40.0 | IQR = 20.0
  fences: -10.0 to 70.0
  actual data range: 10 to 50
  outliers: none

data: [10, 20, 30, 40, 500]
  Q1 = 20.0 | Q3 = 40.0 | IQR = 20.0
  fences: -10.0 to 70.0
  actual data range: 10 to 500
  outliers: [500]



Notice something important in that output.

Both datasets gave the **same Q1, Q3, IQR and fences**. The 500 didn't shift
them at all — because quartiles only care about position, not size.

So the fence stayed honest and *caught* the outlier instead of being dragged
by it. That's what "outlier resistant" means in practice.

Now compare what happens to the standard deviation.

In [8]:
clean_data = [10, 20, 30, 40, 50]
data_with_outlier = [10, 20, 30, 40, 500]

print("standard deviation of", clean_data, "=", round(np.std(clean_data), 2))
print("standard deviation of", data_with_outlier, "=", round(np.std(data_with_outlier), 2))

standard deviation of [10, 20, 30, 40, 50] = 14.14
standard deviation of [10, 20, 30, 40, 500] = 190.26


The standard deviation went from about 14 to about 190 — it blew up completely.

That's because it measures distance from the *mean* (which the outlier already
dragged), uses every single data point, and squares those distances, so extreme
values get amplified even further.

So:

| | measures | survives outliers? |
|---|---|---|
| standard deviation | typical distance from the mean | no |
| IQR | range of the middle 50% | yes |

Same pattern as mean vs median. One is sensitive to extremes, one isn't.

### What to actually do with an outlier
Don't just delete it. Find out why it's there first.

- a data error (age recorded as 999) → fix or remove it
- a genuine rare event (a real billionaire) → keep it, it's real information
- fraud or anomaly detection → the outliers are the whole point, never delete them

### All of this is just df.describe()

When you open a new dataset in pandas, `df.describe()` gives you count, mean,
std, min, 25%, 50%, 75% and max for every column.

Every one of those numbers is something from this notebook.

In [9]:
import pandas as pd

df = pd.DataFrame({"salary": [30, 35, 40, 45, 50, 1000000]})
print(df.describe())

               salary
count        6.000000
mean    166700.000000
std     408231.960593
min         30.000000
25%         36.250000
50%         42.500000
75%         48.750000
max    1000000.000000


Look at that output and you can already spot the problem:

- mean is 166,700 but the median (50%) is only 42.5 → huge gap → data is skewed
- max is 1,000,000 but 75% is only 48.75 → there's an extreme value up top
- std is enormous compared to the middle of the data → confirms it

### Things to check whenever you run describe()

- mean far from the median? → skewed data or outliers
- max much bigger than the 75%? → outliers on the high end
- std tiny compared to the mean? → values are tightly packed
- columns on wildly different scales? → you'll need to normalise before training
